`Keywords`

- Speech Emotion Recognition
- Multilingual Emotion Classification
- Mel-Spectrogram
- Deep Learning
- Hybrid CNN-Transformer Architecture
- ConvNeXt Blocks
- Convolutional Neural Network
- Transformer Encoder
- Attention Mechanism

# Abstract

Speech emotion recognition (SER) remains a challenging problem due to variations in speakers, languages, recording conditions, and dataset-specific biases. So this work presents the design and evaluation of a robust emotion classification system capable of handling diverse, multi-source audio data. A combined dataset of over 10000 audio samples was constructed by integrating 9 publicly available and self-collected datasets, covering 6 emotion categories: anger, disgust, fear, happiness, neutrality, and sadness. All audio samples were preprocessed into fixed-length (8-second) Mel-spectrograms to ensure consistency across sources. The proposed model adopts a hybrid CNN–Transformer architecture. A convolutional backbone, enhanced with ConvNeXt-style blocks, is used to extract spectral features, while a Transformer encoder captures long-range temporal dependencies. An attention-based pooling mechanism is further applied to aggregate temporal information for final classification. The model was trained using stratified splits with label smoothing and evaluated on the test set. The system achieved an overall accuracy of 71.0%, with strong performance on structured datasets such as TESS (100.0%) and RAVDESS Song (92.0%), and relatively balanced results across the two genders. However, performance dropped significantly on more natural, real-world datasets (e.g., Friends and iPartment), highlighting the persistent challenge of cross-domain generalization. These results demonstrate the effectiveness of CNN–Transformer architectures for SER while emphasizing the ongoing challenge of generalization across heterogeneous data sources.

# Introduction

Human speech conveys not only linguistic content but also rich emotional information that reflects a speaker's internal state. The ability to automatically recognize these emotions—commonly referred to as speech emotion recognition (SER)—has become increasingly important in applications such as human-computer interaction, virtual assistants, and mental health assessment. For instance, emotion-aware systems can enable virtual assistants to respond more empathetically, assist in monitoring psychological well-being through vocal cues, and improve user experience in customer service by detecting frustration or satisfaction in real time. Such capabilities also support emerging applications in affective computing, including AI-driven conversational agents designed for companionship, as well as tools that assist therapists and counselors by providing additional insights into a speaker's internal state. As voice-driven technologies continue to expand, enabling systems to interpret emotional cues alongside semantic content is essential for creating more natural, adaptive, and human-centered interactions.

Despite growing interest in SER, the task remains inherently challenging due to variability in speakers, languages, and recording conditions. Differences in vocal characteristics, speaking styles, languages, and background noise can significantly affect model performance and limit generalization. Early work in this area relied on handcrafted acoustic features, such as pitch, energy, and Mel-frequency cepstral coefficients (MFCCs), which were designed to capture prosodic and spectral properties of speech. These features were typically paired with traditional machine learning classifiers, including support vector machines and hidden Markov models [@schuller2009interspeech]. While such approaches laid the groundwork for SER research and provided valuable insights into the acoustic correlates of emotion, their effectiveness was often constrained by the quality and completeness of manually engineered features, which may fail to capture more complex or subtle emotional patterns present in speech.

More recently, deep learning methods have significantly advanced the field by enabling end-to-end feature learning. Convolutional neural networks (CNNs) have been widely adopted for capturing local spectral patterns in time-frequency representations such as spectrograms [@trigeorgis2016adieu]. Recurrent neural networks (RNNs), particularly long short-term memory (LSTM) models, have been used to model temporal dependencies in speech signals [@satt2017efficient]. Building on these advances, Transformer-based architectures have emerged as a powerful alternative, offering improved capability to capture long-range dependencies through self-attention mechanisms [@vaswani2017attention]. Hybrid architectures that combine CNNs with sequence modeling techniques have shown promising results by leveraging both local and global information.

However, a significant limitation in much of the existing literature is the reliance on single datasets, often collected under controlled conditions. Models trained in such settings tend to achieve high accuracy but frequently fail to generalize to new datasets with different characteristics. This issue, commonly referred to as domain shift, poses a major challenge for real-world deployment of SER systems. In addition, many approaches based solely on convolutional neural networks (CNNs), while effective at capturing local spectral patterns, are inherently limited in their ability to model long-range temporal dependencies in speech. Furthermore, simpler CNN architectures may lack the representational capacity needed to extract robust and discriminative features from audio data, which is highly variable and noisy. As a result, such models may struggle to capture both the complex spectral structures and broader temporal dynamics that are critical for accurate emotion recognition, particularly in more natural and variable speech scenarios. Recent studies have begun to explore cross-corpus evaluation and domain adaptation techniques [@latif2021survey], yet achieving robust performance across diverse and heterogeneous data sources remains an open problem.

This study seeks to address these limitations by examining the performance of a unified SER system trained on a diverse collection of audio data. The central research question is: to what extent can a single model learn generalized emotional representations that transfer across different datasets? In particular, this work investigates whether improving both the strength of feature extraction and the modeling of temporal dependencies—through the integration of advanced convolutional architectures and attention-based sequence modeling—can enhance robustness to variability in speech data.

The primary objective of this research is to evaluate the effectiveness of a hybrid deep learning architecture in a cross-domain setting and to analyze performance variations across different types of speech. It is hypothesized that combining a more expressive convolutional backbone for feature extraction with attention-based temporal modeling will improve generalization compared to simpler or single-architecture approaches. Additionally, this study aims to identify key challenges associated with domain shift and provide insights into the limitations of current SER systems.

The remainder of this paper is organized as follows. The next sections describe the methodology and experimental design, followed by a presentation of results and evaluation metrics. Finally, the findings are discussed in relation to existing work, along with implications and directions for future research.

# Theory 

## Overview

The speech emotion recognition (SER) system in this work follows a structured pipeline that transforms raw audio signals into emotion predictions through a sequence of signal processing and deep learning components. The overall framework can be viewed as a mapping:$$y = f(x)$$ where $x$ denotes the raw speech signal and $y \in {0,1,\dots,5}$ represents the predicted emotion label. The function $f(\cdot)$ is composed of multiple stages, including feature extraction, representation learning, temporal modeling, and classification.

## Audio Representation

Given an input audio signal $x(t)$, a time–frequency representation is first computed using the Mel-spectrogram. This transformation maps the 1D waveform into a 2D representation that better reflects human auditory perception.

The Mel-spectrogram is defined as:$$M(f,t)=log(MelFilterBank(∣STFT(x(t))∣^2))$$

where:

* $\text{STFT}(\cdot)$ denotes the short-time Fourier transform
* $|\cdot|^2$ represents the power spectrum
* $\text{MelFilterBank}(\cdot)$ projects frequencies onto the Mel scale

To ensure consistency across datasets, all audio samples are normalized to a fixed duration and standardized:$$\hat{M} = \frac{M-\mu}{\sigma+\epsilon}$$ where $\mu$ and $\sigma$ are the mean and standard deviation of the spectrogram.

## Feature Extraction

The normalized Mel-spectrogram $\hat{M}$ is passed through a convolutional feature extractor:$$H = f_{\mathrm{CNN}}(\hat{M})$$

This stage captures local spectral patterns such as pitch variations, formants, and energy distributions. Compared to simple convolutional layers, more expressive architectures enable the model to learn richer and more robust representations from diverse and noisy speech inputs.

## Temporal Modeling

Speech signals contain strong temporal dependencies, as emotional cues often unfold over time. To model these dependencies, the extracted feature sequence is processed using a sequence encoder:$$Z = f_{\mathrm{seq}}(H)$$

In this work, attention-based sequence modeling is employed to capture long-range relationships across time steps. This allows the model to focus on emotionally salient regions of the speech signal, rather than treating all frames equally.

## Attention-Based Aggregation

To aggregate temporal information into a fixed-length representation, an attention mechanism is applied:$$\alpha_t = \frac{\exp\left( \mathbf{w}^\top \mathbf{z}_t \right)}{\sum _k \exp\left( \mathbf{w}^\top \mathbf{z}_k \right)}$$ $$\mathbf{z} = \sum\limits_{t=1}^{T} \alpha_t \mathbf{z}_t$$

where:

$z_t$ is the feature at time step $t$
$\alpha_t$ represents the attention weight
$z$ is the aggregated representation

This mechanism enables the model to emphasize important emotional segments in the speech signal.

## Classification Objective

The final representation $z$ is passed to a classifier to produce class probabilities:$$\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{W}\mathbf{z} + \mathbf{b})$$

The model is trained using cross-entropy loss with label smoothing:$$\mathcal{L} = - \sum_{i=1}^{C} y_i^{(s)} \log \left( \hat{y}_i \right)$$

where $y^{(s)}$ is the smoothed label distribution.

## Overall Pipeline

The overall processing pipeline of the proposed SER system is illustrated below:

```{mermaid}
flowchart TD
    A[Raw Audio Data] --> B[Mel-Spectrogram]

    subgraph M[Hybrid Deep Learning Model]
        C[CNN Feature Extractor] --> D[Temporal Modeling] --> E[Attention Pooling]
    end

    B --> C
    E --> F[Emotion Prediction]
```

# Methods

This section describes the datasets used in this study, the preprocessing pipeline applied to raw audio signals, and the proposed deep learning architecture. The overall goal is to build a robust speech emotion recognition system capable of generalizing across multiple datasets, languages, and recording conditions.

## Datasets

To improve robustness and cross-domain generalization, this study combines multiple speech emotion datasets. These datasets differ significantly in language, speaker demographics, recording quality, and emotional expression style, making the task more challenging but also more realistic.

***RAVDESS (Speech)***

The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS) contains recordings from 24 professional North American actors. The dataset is balanced across gender and is widely used as a benchmark for SER tasks due to its high recording quality and controlled conditions. In this work, both speech and song subsets are included.

***RAVDESS (Song)***

This subset contains sung emotional expressions from the same actors as RAVDESS speech. Singing introduces additional variations in pitch, rhythm, and prosody, making emotion recognition more challenging. Including this dataset helps evaluate robustness to non-speech vocal expressions.

***CREMA-D***

CREMA-D is a large emotional speech dataset consisting of recordings from 91 actors. Each actor produces multiple utterances across 6 emotions with varying intensity levels. Compared to RAVDESS, CREMA-D exhibits more variability in speaking style and emotional expression, making it more representative of real-world scenarios. Due to class imbalance in the original dataset, a controlled downsampling strategy was applied to balance emotion and gender distribution.

***TESS***

The Toronto Emotional Speech Set (TESS) contains high-quality recordings from two female actors speaking target words embedded in a carrier phrase. Although highly clean and well-balanced, TESS has limited speaker diversity. To prevent bias, balanced sampling across actors and emotion categories was performed.

***EmoDB***

The Berlin EmoDB dataset is a German emotional speech corpus recorded by 10 professional actors. It is known for its clear emotional expression and relatively small size. EmoDB provides strong performance signals due to its high-quality labeling, but also limited speaker diversity.

***ShEMO***

ShEMO is a Persian emotional speech dataset consisting of semi-natural utterances from 87 native speakers. Unlike acted datasets, ShEMO contains more spontaneous and less exaggerated emotional expressions, making it valuable for evaluating real-world robustness.

***EMOVO***

EMOVO is an Italian emotional speech dataset recorded by professional actors. It provides additional language diversity, improving multilingual generalization.

***Friends*** (Self-Collected English TV Data)

This dataset consists of dialogue segments extracted from the TV series Friends. It contains natural conversational speech with emotional annotations. Compared to acted datasets, it includes background noise, overlapping speech, and informal dialogue, making it significantly more challenging and realistic.

***iPartment*** (Self-Collected Chinese TV Data)

This dataset is collected from the Chinese TV series iPartment. Similar to the Friends dataset, it contains real-world conversational speech in Mandarin Chinese with emotional labels. It introduces additional linguistic diversity and natural emotional variation.

## Data Preprocessing

In the data preprocessing section, all audio samples are transformed into fixed-length Mel-spectrogram representations to ensure a consistent and structured input format for the model. This preprocessing pipeline standardizes recordings from different datasets, which vary in sampling rates, duration, and recording conditions.

Each audio signal is first resampled to 22050 Hz to maintain uniform temporal resolution. Silence segments are then removed using amplitude-based trimming to reduce the influence of non-informative regions. After trimming, each waveform is adjusted to a fixed duration of 8 seconds: signals longer than this threshold are truncated, while shorter signals are zero-padded.

Following temporal normalization, a Mel-spectrogram is computed using 128 Mel frequency bins, with a Fast Fourier Transform (FFT) window size of 1024 and a hop length of 512. The resulting power spectrogram is converted into the logarithmic decibel (dB) scale to better reflect human auditory perception. To stabilize training and reduce variance across samples, each spectrogram is normalized using per-sample standardization (zero mean and unit variance).

To improve computational efficiency, all processed Mel-spectrograms are precomputed and stored as cached .npy files, avoiding repeated processing during training.

Data augmentation is also applied during training to enhance model robustness and reduce overfitting. Specifically, the SpecAugment technique is employed, which introduces random perturbations directly on the Mel-spectrograms. Two types of masking are applied:
(1) frequency masking, which randomly removes contiguous frequency bands, simulating variations in vocal characteristics
(2) time masking, which randomly removes temporal segments, mimicking missing or corrupted speech signals

These augmentations are applied dynamically during training only, while validation and test data remain unaltered to ensure fair evaluation.

## Model Architecture

The proposed model adopts a hybrid CNN–Transformer architecture designed to effectively capture both local spectral patterns and long-range temporal dependencies in speech signals. The model takes Mel-spectrograms as input and processes them through three main components: a convolutional feature extractor, a Transformer-based temporal encoder, and an attention-based classification head.

### Convolutional Feature Extractor

The first stage of the model is a convolutional backbone inspired by modern architectures such as ConvNeXt. Its primary purpose is to extract high-level representations from the input Mel-spectrogram while preserving temporal structure.

The input spectrogram, with dimensions (1 × frequency × time), is first passed through a stem convolution layer that increases the channel dimension to 32, followed by Group Normalization and GELU activation. Compared to traditional Batch Normalization, Group Normalization is less sensitive to batch size and provides more stable training in this setting.

The backbone then consists of multiple ConvNeXt-style blocks, each designed to improve feature extraction efficiency. Each block includes:

- A depthwise convolution (kernel size 7×7) to capture local spatial patterns across time–frequency regions,
- A normalization layer (GroupNorm) to stabilize feature distributions,
- A pointwise convolution-based MLP that expands and compresses channel dimensions, enabling richer feature interactions,
- A residual connection to facilitate gradient flow and prevent degradation in deeper networks.

Between stages, 1×1 convolution layers are used to increase the number of channels (32 -> 64 -> 128 -> 256), allowing the network to learn progressively more abstract representations. Additionally, max pooling is applied only along the frequency axis, reducing spectral resolution while preserving temporal resolution. This design choice ensures that temporal dynamics, which are critical for emotion recognition, remain intact.

Overall, this module transforms the input spectrogram into a compact, high-dimensional feature map encoding both spectral and local temporal information.

### Temporal Modeling with Transformer

After convolutional processing, the feature map is reshaped into a sequence of feature vectors along the time dimension. Each time step is treated as a token representing the acoustic features at that moment.

These features are projected into a fixed embedding space and passed through a Transformer encoder composed of two layers with multi-head self-attention. The Transformer enables the model to capture long-range dependencies and contextual relationships across the entire utterance, which are essential for understanding emotional dynamics that evolve over time.

To preserve the temporal order of the sequence, positional encoding is added to the input embeddings. Layer normalization is applied after the Transformer to stabilize training and improve convergence.

### Attention-Based Pooling and Classification

Instead of using simple pooling methods (e.g., average pooling), the model employs an attention-based pooling mechanism to aggregate temporal features. This mechanism learns a set of weights over time steps, allowing the model to focus on the most emotionally informative segments of the speech signal.

The weighted sum of the Transformer outputs produces a fixed-length representation, which is then passed through a dropout layer to reduce overfitting. Finally, a fully connected layer maps the representation to the six emotion classes.

This attention-based aggregation enables the model to selectively emphasize critical temporal regions, improving classification performance, particularly in longer or noisier utterances.

## Training Procedure

The model is trained using the cross-entropy loss function with label smoothing (0.1) to improve generalization. Optimization is performed using the Adam optimizer with an initial learning rate of 3e-4 and weight decay of 1e-4. A ReduceLROnPlateau scheduler is used to reduce the learning rate when validation loss plateaus. Gradient clipping (max norm = 1.0) is applied to stabilize training.

Early stopping is employed with a patience of 5 epochs based on validation loss. The model achieving the best validation performance is saved and used for final evaluation.

# Results

This section presents the performance of the proposed model on the test set, including overall accuracy, class-wise performance, and detailed evaluations across datasets, gender, and emotion categories.

## Overall Performance

The proposed CNN–Transformer model achieved a test accuracy of 71.02% on the combined multi-dataset test set. During training, the model showed steady improvement in both training and validation accuracy, with early stopping triggered after 16 epochs to prevent overfitting. The best validation performance corresponded closely with the final test accuracy, indicating stable generalization.

## Classification Performance by Emotion

A detailed classification report and a confusion matrix were generated to evaluate performance across the six emotion classes:

<center>

|   Emotion   | Precision  |  Recall  | F1-score  | Support |
|------------:|:----------:|:--------:|:----------:|:------:|
|      Angry  |       0.75 |    0.89  |    0.81    |   298 |
|     Disgust |      0.62  |    0.79  |    0.69    |   229 |
|       Fear  |    0.63    |  0.73   |   0.67      | 238 |
|      Happy  |    0.78    |  0.44   |   0.56      | 233 |
|    Neutral  |    0.75    |  0.83   |   0.79      | 292 |
|        Sad  |    0.82    |  0.50    |  0.63      | 242 |
|   ---------  |      ---       |    ---      | ---     | --- |
|  macro avg  |    0.72    |  0.70     | 0.69     | 1532 |
|weighted avg |     0.73   |   0.71    |  0.70    |  1532 |

: Classification Report

</center>

<br>
<br/>

<center>

![Confusion Matrix](./images/cm_whole.png){width=70%}

</center>

The model achieves its strongest performance on angry (F1-score: 0.81) and neutral (F1-score: 0.79). These results can be attributed to the relatively distinct acoustic characteristics of these emotions. Angry speech typically exhibits high energy, increased pitch, and sharper spectral variations, making it easier for the model to distinguish. Similarly, neutral speech tends to have stable and less variable acoustic patterns, which may lead to more consistent representations across datasets.

Moderate performance is observed for disgust (F1-score: 0.69) and fear (F1-score: 0.67). According to the confusion matrix, these emotions share overlapping acoustic features with each other, which can lead to confusion during classification. Additionally, variations in how these emotions are expressed across different datasets may reduce consistency in learned representations.

Lower performance is observed for sad (F1-score: 0.63) and particularly happy (F1-score: 0.56). The reduced recall for happy suggests that the model frequently misclassifies this emotion as others. Only 103 samples are correctly classified as happy, while a large number are misclassified as angry (42), disgust (33), or fear (33). This dispersion indicates that happy speech does not form a well-defined cluster and overlaps with both high-energy and neutral-like expressions. Similarly, sad shows substantial confusion with neutral (35), disgust (37), and fear (33). This suggests that low-arousal emotions are harder to distinguish, particularly when their acoustic features—such as reduced energy and slower tempo—are similar.

The confusion between sad and neutral, as well as between happy and multiple other classes, highlights a broader pattern: the model struggles more with subtle or low-intensity emotions, where acoustic differences are less pronounced. In contrast, high-arousal and acoustically distinct emotions (e.g., angry) are more reliably classified.

Overall, the results suggest that the model more effectively captures high-arousal and acoustically distinctive emotions, while struggling with low-arousal or subtly expressed emotions that exhibit greater overlap in their acoustic features.

## Performance by Gender

The model achieved:

- 73.86% accuracy for female speakers
- 68.19% accuracy for male speakers

This indicates a slight performance advantage in female speech, although the difference is relatively small. Performance trends were generally consistent across datasets, with no extreme gender imbalance observed.

## Performance Across Datasets

Model performance varied substantially across datasets, reflecting differences in recording conditions, speaker variability, language, and emotional expressiveness:

- TESS (100%) achieved perfect accuracy, which can likely be attributed to its highly controlled recording environment, limited speaker set, and exaggerated emotional expressions. These characteristics reduce variability and make emotion boundaries more separable for the model.

- RAVDESS Song (92.00%) also demonstrated strong performance. Singing tends to amplify prosodic features such as pitch, rhythm, and intensity, which are key cues for emotion recognition. This structured and expressive nature makes emotional patterns more distinguishable compared to natural speech.

- EmoDB (78.26%) and ShEMO (76.02%) showed solid performance despite being in different languages (German and Persian, respectively). This suggests that the model successfully captures some language-independent acoustic features (e.g., tone, energy, and temporal dynamics), although some performance degradation may occur due to phonetic and cultural differences in emotional expression.

- RAVDESS (speech) (66.67%) achieved moderate accuracy. Compared to its singing counterpart, spoken utterances in RAVDESS are less exaggerated and exhibit more subtle emotional variations, making classification more challenging.

- CREMA-D (57.79%) and iPartment (56.57%) showed lower performance, likely due to greater diversity in speakers, recording conditions, and emotional delivery styles. These datasets contain more natural variability, which increases overlap and reduces model generalization.

- EMOVO (50.00%) presented moderate difficulty. While it is an acted dataset, differences in language (Italian) and possibly less consistent emotional intensity may contribute to reduced performance.

- Friends (33.33%) had the lowest accuracy, highlighting the challenge of real-world conversational data. Unlike controlled datasets, this dataset contains spontaneous speech, background noise, overlapping dialogue, and subtle or mixed emotions, all of which significantly increase ambiguity and reduce classification performance. Compared with the iPartment series released after 2010, Friends was recorded at an earlier time, resulting in poorer sound quality, which partially explains the disparity between the performance on the two real-world datasets.

Overall, these results suggest that the model performs best on clean, well-structured, and highly expressive datasets, where emotional cues are clear and consistent. In contrast, performance degrades on datasets with greater variability, naturalistic speech, and less distinct emotional boundaries, underscoring the challenges of real-world speech emotion recognition.

## Performance by Dataset and Emotion

A more fine-grained analysis reveals substantial variation in recognition accuracy across emotion categories within each dataset, highlighting both consistent trends and dataset-specific challenges.

Across most datasets, high-arousal emotions such as angry and disgust tend to achieve relatively strong performance, whereas low-arousal or more subtle emotions such as happy and sad exhibit significantly lower and more variable performance. The neutral class shows a consistent pattern of strong performance across many datasets. These are consistent with the conclusion we reached above.

Several dataset-specific anomalies further illustrate the challenges of cross-domain emotion recognition:

- ShEMO exhibits a complete failure on fear (0.00), despite strong performance on angry and neutral. This suggests a domain mismatch, where the acoustic realization of some emotions in Persian speech differs significantly from other datasets.
- Friends (real-world data) shows highly uneven performance, with relatively strong results for angry (0.89) and neutral (0.78), but zero accuracy for happy and very low performance for other emotions. This reinforces the difficulty of handling spontaneous, conversational speech, where emotions are often subtle, mixed, or context-dependent.
- RAVDESS Song maintains consistently high performance across all emotions, further supporting the idea that structured and exaggerated vocal expressions enhance model discriminability.

---------

**Overall**, these results suggest that emotion recognition performance is not only emotion-dependent but also highly dataset-dependent. Emotions with clear and strong acoustic signatures are more reliably classified, while those with subtle or overlapping characteristics remain challenging—especially in datasets with greater linguistic diversity or naturalistic recording conditions.

# Conclusions

This study investigated whether a single speech emotion recognition (SER) model can learn meaningful and generalizable emotional patterns across diverse datasets. By combining multiple speech corpora and applying a hybrid CNN–Transformer architecture, the model achieved an overall accuracy of 71.02%, showing that it is possible to extract useful emotional representations from heterogeneous audio data. Results show strong performance on structured, high-quality datasets, but noticeable degradation on real-world conversational speech, highlighting the ongoing challenge of generalization.

Key findings indicate that emotions with clear acoustic patterns, such as anger and neutral, are more reliably recognized, while more subtle emotions like happiness and sadness remain difficult due to overlapping features. Additionally, dataset characteristics—such as recording quality, speaker diversity, and expression style—have a significant impact on performance. These findings emphasize that, while modern deep learning architectures are powerful, generalization across domains remains a central challenge in SER.

Despite these contributions, this study has several limitations. The combined datasets vary significantly in quality, annotation standards, and recording conditions, which introduces inconsistencies that make learning a unified representation more difficult. Additionally, some datasets contain only a small number of speakers, limiting diversity and reducing how well the model reflects real-world scenarios. From a practical perspective, the computational cost of training more advanced models also posed a constraint. The hybrid architecture requires substantial training time on a standard laptop GPU, making extensive hyperparameter tuning and experimentation difficult.

Future work can build on this research in several directions. Improving data quality and consistency could enhance cross-dataset generalization. Incorporating more natural and diverse real-world data would also make the model more robust in practical applications. On the modeling side, integrating additional modalities, such as text or facial expressions, may provide complementary information and further improve emotion recognition accuracy.

In summary, this work demonstrates both the potential and the current limitations of deep learning approaches for speech emotion recognition. While meaningful progress has been made, achieving reliable performance across real-world conditions remains an open and important challenge.

## References

::: {#refs}
:::